# Law of total variance

_Probability & Statistics_

**Split a variable's spread into two parts: scatter inside each group, plus scatter between the groups.**

Imagine the data is sorted into groups by a second variable $Y$.

     The total spread of $X$ comes from two sources.

---

This notebook builds the variance decomposition one step at a time and checks it against a simulated mixture. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

### Step 1 — Simulate a two-group mixture

We build a population split into two equal groups (think two classes of students). The groups have means $70$ and $80$ but share the same within-group variance of $25$. For each of a million people we pick a group $Y$, then draw a score $X$ from that group's normal distribution.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Two equal classes. Class means and shared within-group variance.
means = np.array([70.0, 80.0])
within_var = 25.0

n = 1_000_000
grp = rng.integers(0, 2, n)                       # which class (Y)
x = rng.normal(means[grp], np.sqrt(within_var))   # score (X)

### Step 2 — Compute the two formula terms

The law of total variance says $\operatorname{Var}(X) = E[\operatorname{Var}(X\mid Y)] + \operatorname{Var}(E[X\mid Y])$. The **within** term is the average of each group's internal variance — here just the shared value $25$. The **between** term is the variance of the group means around the grand mean. Their sum is the predicted total.

In [ ]:
# Formula: within + between.
within = within_var                          # E[Var(X|Y)]

grand = means.mean()
between = np.mean((means - grand) ** 2)      # Var(E[X|Y]), equal weights

formula_total = within + between

print("within  E[Var(X|Y)] =", within)
print("between Var(E[X|Y]) =", between)
print("formula total       =", formula_total)

### Step 3 — Check against the raw simulated variance

If the decomposition is right, the plain variance of the whole mixture should equal within + between. We measure the raw variance of all the simulated scores and compare it to the formula total of $50$.

In [ ]:
# Monte-Carlo: raw variance of the mixture should match.
print("MC Var(X)           =", round(x.var(), 2), "(theory 50)")

## Visualize the data & results

_Does within-group plus between-group variance add up to the total variance of the mixture?_

### Step 1 — Rebuild the mixture and the decomposition

To make the visualization self-contained, we regenerate the same two-group mixture and recompute the within term, the between term, the formula total, and the raw Monte-Carlo variance. These are the four numbers we will compare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Law of total variance: Var(X) = E[Var(X|Y)] + Var(E[X|Y]).
# Two equal classes, means 70/80, shared within-group variance 25.
rng = np.random.default_rng(0)
means = np.array([70.0, 80.0])
within_var = 25.0

n = 1_000_000
grp = rng.integers(0, 2, n)
x = rng.normal(means[grp], np.sqrt(within_var))

within = within_var                              # E[Var(X|Y)]
between = np.mean((means - means.mean()) ** 2)   # Var(E[X|Y]), equal weights
total = within + between                         # formula total
mc = x.var()                                     # raw simulated variance

### Step 2 — Chart within + between against the total

Finally we bar-chart the four quantities. The first two bars (within and between) should stack up to the third (formula total), and the fourth (Monte-Carlo variance of the mixture) should match it — visual proof that the spread really does split cleanly into within-group plus between-group parts.

In [ ]:
labels = ['within E[Var(X|Y)]', 'between Var(E[X|Y])', 'formula total', 'Monte-Carlo Var(X)']
vals = [within, between, total, mc]
colors = ['#7ee787', '#c89bff', '#ffb454', '#4ea1ff']

plt.figure(figsize=(8, 4))
plt.bar(labels, vals, color=colors)
plt.title('Variance decomposition: within + between = total')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()